In [45]:
import pandas as pd
training_data = pd.read_csv('data/train.csv')
training_data.head()
print(training_data.shape)
training_data.head()

(221, 37)


,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


In [46]:
from lifelines import CoxPHFitter
data_iteration1 = training_data.copy()
TARGETS = ['event', 'time_to_hit_hours']
features = [
    'area_first_ha',
    'area_growth_abs_0_5h',
    'area_growth_rel_0_5h',
    'centroid_displacement_m',
    'centroid_speed_m_per_h',
    'spread_bearing_sin',
    'spread_bearing_cos',
    'num_perimeters_0_5h',
    'dt_first_last_0_5h',
    'low_temporal_resolution_0_5h'
]
data_iteration1_model = data_iteration1[TARGETS + features].copy()
cph = CoxPHFitter()
cph.fit(data_iteration1_model, duration_col='time_to_hit_hours', event_col='event')
print(cph.summary)

                                  coef  exp(coef)  se(coef)  coef lower 95%  \
covariate                                                                     
area_first_ha                -0.000483   0.999517  0.000272       -0.001015   
area_growth_abs_0_5h          0.002599   1.002602  0.001633       -0.000601   
area_growth_rel_0_5h          0.281192   1.324708  0.231271       -0.172091   
centroid_displacement_m      -0.001429   0.998572  0.003333       -0.007962   
centroid_speed_m_per_h       -0.007499   0.992529  0.012521       -0.032040   
spread_bearing_sin            0.422937   1.526437  0.382353       -0.326462   
spread_bearing_cos           -0.570126   0.565454  0.332942       -1.222680   
num_perimeters_0_5h           0.093886   1.098434  0.059653       -0.023033   
dt_first_last_0_5h            0.034711   1.035320  0.153073       -0.265308   
low_temporal_resolution_0_5h -0.873873   0.417332  0.621823       -2.092623   

                              coef upper 95%  exp(c

In [47]:
testing_data = pd.read_csv('data/test.csv')
print(testing_data.shape)
X_test = testing_data[features]
survival_predictions_test = cph.predict_survival_function(X_test)
# extract probabilities at desired horizons
time_horizons = [12, 24, 48, 72]
submission_data = []
survival_predictions_test.head()

(95, 35)


,0,1,2,3,4,5,6,7,8,9,...,85,86,87,88,89,90,91,92,93,94
0.001220,0.997846,0.997977,0.997851,0.998121,0.979656,0.998091,0.997848,0.999713,0.998669,0.997871,...,0.997883,0.970840,0.998864,0.965361,0.997925,0.993432,0.998000,0.997933,0.993454,0.997860
0.007176,0.995679,0.995940,0.995688,0.996230,0.959557,0.996170,0.995682,0.999424,0.997328,0.995729,...,0.995753,0.942293,0.997720,0.931641,0.995837,0.986851,0.995987,0.995853,0.986895,0.995706
0.064361,0.993513,0.993904,0.993526,0.994340,0.939838,0.994249,0.993517,0.999134,0.995987,0.993588,...,0.993624,0.914539,0.996575,0.899045,0.993749,0.980303,0.993975,0.993774,0.980369,0.993554
0.130320,0.991339,0.991861,0.991357,0.992442,0.920417,0.992321,0.991345,0.998843,0.994640,0.991440,...,0.991487,0.887453,0.995425,0.867417,0.991654,0.973762,0.991955,0.991688,0.973849,0.991394
0.132979,0.989105,0.989761,0.989127,0.990491,0.900828,0.990339,0.989112,0.998543,0.993255,0.989231,...,0.989291,0.860388,0.994242,0.835995,0.989501,0.967069,0.989879,0.989543,0.967178,0.989173


In [48]:
import numpy as np
def generate_predictions(survival_predictions):
    for col in survival_predictions_test.columns:
        survival_probs = np.interp(
            time_horizons,
            survival_predictions_test.index.values,
            survival_predictions_test[col].values
        )
        prob_hit = 1 - survival_probs
        submission_data.append(prob_hit)
    submission_df = pd.DataFrame(submission_data, columns=['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h'])
    # Add event_id
    submission_df['event_id'] = testing_data['event_id']

    # Reorder columns if needed
    submission_df = submission_df[['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']]
    return submission_df

In [49]:
submission = generate_predictions(survival_predictions_test)
submission.to_csv('baseline_submission.csv', index=False)